# 09_make_descriptors — descriptor 217종 계산

**한 줄 요약:** 실측 화합물(3번째 시트)에 대해 RDKit **descriptor(물성 수치) 217종**을 계산하고 라벨과 함께 저장한다. (이후 descriptor 학습·해석의 재료)
**큰 흐름:** ① 준비·라벨링·읽기 → ② descriptor 계산 → ③ 결측 정리·저장

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 + 라벨링 + 데이터 읽기
라이브러리를 가져오고, IC50로 active/inactive 라벨을 붙이며 3번째 시트를 읽는다.

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

SRC = "data/HSD17B13_IC50_merged.xlsx"
OUT_XLSX = "data/HSD17B13_descriptors.xlsx"
OUT_CSV = "data/HSD17B13_descriptors.csv"
ACTIVE_MAX = 10000.0


def make_label(ic50, rel):
    rel = str(rel).strip()
    if pd.isna(ic50):
        return np.nan
    if rel in ("<", "<="):
        return 1 if ic50 <= ACTIVE_MAX else 0
    if rel in (">", ">="):
        return 0
    return 1 if ic50 <= ACTIVE_MAX else 0


df = pd.read_excel(SRC, sheet_name="same_dedup_keepdiff")
df = df.dropna(subset=["canonical_smiles"]).reset_index(drop=True)
df["label"] = [make_label(v, r) for v, r in zip(df["ic50_nM"], df["relation"])]

🔎 **코드 뜯어보기 (셀 1)** *(make_label은 05, read_excel은 01에서 설명)*
- `from rdkit.Chem import Descriptors` : descriptor 계산 함수 모음.

### 셀 2 — descriptor 217종 계산
각 분자에 대해 217개 물성 값을 계산해 표로 만든다.

In [ ]:
desc_names = [name for name, _ in Descriptors._descList]
print(f"RDKit descriptor {len(desc_names)}종 계산")

meta_cols = ["canonical_smiles", "ic50_nM", "relation", "sources", "label"]
rows, keep_idx = [], []
for i, smi in enumerate(df["canonical_smiles"]):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    d = Descriptors.CalcMolDescriptors(mol)          # 전체 descriptor dict
    rows.append([d.get(n, np.nan) for n in desc_names])
    keep_idx.append(i)

X = pd.DataFrame(rows, columns=desc_names)

🔎 **코드 뜯어보기 (셀 2)**
- `desc_names = [name for name, _ in Descriptors._descList]` : 등록된 descriptor **이름 217개**를 리스트로.
- `Descriptors.CalcMolDescriptors(mol)` : 분자 하나의 217개 값을 딕셔너리로 계산.
- `[d.get(n, np.nan) for n in desc_names]` : 이름 순서대로 값 뽑기(없으면 빈 값). `pd.DataFrame(rows, columns=...)`=표로.

### 셀 3 — 결측/무한대 정리 후 저장
계산값에 이상치가 있는지 확인하고, 분자정보와 합쳐 Excel·CSV로 저장한다.

In [ ]:
n_inf = np.isinf(X.to_numpy(dtype=float, na_value=np.nan)).sum()
X = X.replace([np.inf, -np.inf], np.nan)
n_nan = int(X.isna().sum().sum())
print(f"정상 변환 {len(keep_idx)}/{len(df)}개 | inf {int(n_inf)}개, NaN {n_nan}개 발견 "
      f"(ML 시 imputation/스케일링 필요)")

meta = df.loc[keep_idx, meta_cols].reset_index(drop=True)
out = pd.concat([meta, X.reset_index(drop=True)], axis=1)

out.to_csv(OUT_CSV, index=False)
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    out.to_excel(w, sheet_name="descriptors", index=False)

lab = out["label"].dropna()
print(f"\n저장: {OUT_XLSX} | {OUT_CSV}")
print(f"행 {len(out)} x 열 {out.shape[1]} (메타 {len(meta_cols)} + descriptor {len(desc_names)})")
print(f"라벨 분포: active {int((lab==1).sum())} / inactive {int((lab==0).sum())}")
print("\ndescriptor 예시(앞 12종):", ", ".join(desc_names[:12]))

🔎 **코드 뜯어보기 (셀 3)**
- `np.isinf(...)` : 무한대인지 확인. `.replace([np.inf,-np.inf], np.nan)` : 무한대를 빈 값으로.
- `.isna().sum().sum()` : 빈 값 총 개수. `pd.concat([meta, X], axis=1)` : 분자정보 + descriptor 좌우 결합.
- `.to_csv(...)` / `.to_excel(...)` : 두 형식으로 저장.